<style>
    @import url('https://fonts.googleapis.com/css2?family=Oswald:wght@400;600&display=swap');
    h1.course-title {
        font-family: 'Oswald', sans-serif;
        font-size: 2.4em;
        color: #E7C173;
        letter-spacing: 0.05em;
        border-bottom: 2px solid #E7C173;
        padding-bottom: 0.3em;
        margin-bottom: 0.2em;
    }
    h2.course-subtitle { font-family: 'Oswald', sans-serif; color: #aaa; font-size: 1.2em; }
</style>

<h1 class='course-title'>MACHINE LEARNING IN INDUSTRY</h1>
<h2 class='course-subtitle'>Cardo AI · MSCA Digital Doctoral Network · Day 4</h2>

# Day 4, Block 3 — Data Drift and Model Monitoring with NannyML

## Table of Contents
1. [Scope and Success Criteria](#1-scope)
2. [Why Monitoring? The Silent Decay Problem](#2-why)
3. [Drift Taxonomy](#3-taxonomy)
4. [Setup — Imports and Data](#4-setup)
5. [Reference and Analysis Sets](#5-reference-analysis)
6. [CBPE — Performance Estimation Without Ground Truth](#6-cbpe)
7. [Univariate Drift Detection](#7-univariate)
8. [Multivariate Drift Detection](#8-multivariate)
9. [Simulating Drift](#9-simulation)
10. [Logging Reports to MLflow](#10-mlflow)
11. [Acceptance Checks](#11-checks)

---
## 1. Scope and Success Criteria <a id='1-scope'></a>

By the end of this notebook you should be able to:

- Explain the difference between **covariate shift**, **concept drift**, and **prior probability shift**
- Set up NannyML **reference** and **analysis** sets correctly
- Interpret **CBPE** output — what does an estimated AUC drop mean, and when should you act?
- Identify drifting features using **univariate drift detection**
- Log NannyML reports as **MLflow artifacts** so monitoring history is tracked alongside model history

> **Time budget:** Block 3 (≈45 minutes). The MLflow server from Block 2 should still be running.

---
## 2. Why Monitoring? The Silent Decay Problem <a id='2-why'></a>

Consider a loan default model trained in 2019 on income, employment, and credit history data. In 2020–2021, employment patterns shift dramatically: furlough schemes appear, gig economy income collapses, remote work becomes standard. The model's *feature distribution* has moved — but nobody retrained it.

The model doesn't throw an error. It keeps returning predictions. They just become progressively less accurate. You might only notice six or twelve months later, when loans start defaulting at unexpected rates and you're trying to explain to a regulator why the model drifted out of tolerance.

**The cost of late detection is asymmetric.** Running a drift check costs a few seconds of compute. Missing a distributional shift in a credit risk model costs capital and regulatory credibility.

> 📖 **Recommended reading:** [Failing Loudly (Rabanser et al., 2019)](https://arxiv.org/abs/1810.11953) — a systematic empirical study of drift detection methods. The title says it all: a production ML system should *fail loudly*, not silently.

---
## 3. Drift Taxonomy <a id='3-taxonomy'></a>

There are three fundamentally different things that can go wrong:

| Drift Type | Formal Definition | Credit Risk Example | Detectable by NannyML? |
|---|---|---|---|
| **Covariate shift** | P(X) changes, but P(Y\|X) stays the same | Age distribution of new applicants shifts younger after a marketing campaign | ✅ Univariate + multivariate drift |
| **Concept drift** | P(Y\|X) changes — the relationship between features and outcome changes | Remote work makes `commute_miles` irrelevant for income prediction | ⚠️ CBPE (indirectly) — requires ground truth for confirmation |
| **Prior probability shift** | P(Y) changes — the base rate of the outcome shifts | Recession increases the default rate | ✅ CBPE tracks estimated vs actual performance |

**Key insight:** CBPE is most directly useful for covariate shift and prior probability shift. Concept drift is harder — it requires ground truth labels to confirm. NannyML can raise the alarm that *something* has changed; confirming *why* requires domain knowledge.

---
## 4. Setup — Imports and Data <a id='4-setup'></a>

In [1]:
# Guard: check the MLflow server is reachable
import urllib.request, urllib.error
MLFLOW_URI = "http://127.0.0.1:5000"
try:
    urllib.request.urlopen(f"{MLFLOW_URI}/health", timeout=2)
    print(f"✓ MLflow server is running at {MLFLOW_URI}")
except urllib.error.URLError:
    print(f"⚠  MLflow server not reachable at {MLFLOW_URI}")
    print("   Run: make -f day4/Makefile mlflow-server")

✓ MLflow server is running at http://127.0.0.1:5000


In [2]:
import sys, json, tempfile
from pathlib import Path

import mlflow
import mlflow.sklearn
import nannyml as nml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

repo_root = Path.cwd().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from day4.src.train import (
    DATA_PATH, SEED, TARGET_BIN_COL,
    build_pipeline, get_feature_columns, load_data, split_data,
)

MONITORING_EXPERIMENT = "adult-income-monitoring"
MODEL_NAME = "adult-income-classifier"

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(MONITORING_EXPERIMENT)

np.random.seed(SEED)
print("Imports OK")

Imports OK


In [3]:
# Load data and train a model (same as notebook 01)
data_path = repo_root / DATA_PATH
df = load_data(data_path)
train_df, val_df, test_df = split_data(df)

numeric_cols, categorical_cols = get_feature_columns(train_df)
feature_cols = numeric_cols + categorical_cols

X_train, y_train = train_df[feature_cols], train_df[TARGET_BIN_COL]

params = {"n_estimators": 300, "learning_rate": 0.1, "max_depth": 5}
pipe = build_pipeline(numeric_cols, categorical_cols, **params)
pipe.fit(X_train, y_train)

actual_auc = roc_auc_score(
    test_df[TARGET_BIN_COL],
    pipe.predict_proba(test_df[feature_cols])[:, 1],
)
print(f"Model trained.  Actual test AUC: {actual_auc:.4f}")

Model trained.  Actual test AUC: 0.9166


---
## 5. Reference and Analysis Sets <a id='5-reference-analysis'></a>

NannyML's architecture centres on two datasets:

- **Reference set** — data the model was trained or validated on. Ground truth *is* available. NannyML uses this to learn the probability-to-performance mapping.
- **Analysis set** — production data. Ground truth may *not yet* be available (loans haven't defaulted or repaid yet). NannyML applies the learned mapping to estimate performance.

Both sets need: the feature columns, the model's predicted probabilities, and a `timestamp` column for chunk-based analysis.

In [4]:
# Reference = training split (ground truth available)
reference_df = df[df["split"] == "train"][feature_cols + [TARGET_BIN_COL]].copy()
reference_df["y_pred_proba"] = pipe.predict_proba(reference_df[feature_cols])[:, 1]
reference_df["y_pred"] = (reference_df["y_pred_proba"] >= 0.5).astype(int)
reference_df["timestamp"] = pd.date_range("2023-01-01", periods=len(reference_df), freq="h")

# Analysis = test split (simulate: ground truth withheld from NannyML)
analysis_df = test_df[feature_cols].copy()
analysis_df["y_pred_proba"] = pipe.predict_proba(analysis_df)[:, 1]
analysis_df["y_pred"] = (analysis_df["y_pred_proba"] >= 0.5).astype(int)
analysis_df["timestamp"] = pd.date_range("2024-01-01", periods=len(analysis_df), freq="h")
# Keep a separate copy WITH ground truth for validation at the end
analysis_with_gt = analysis_df.copy()
analysis_with_gt[TARGET_BIN_COL] = test_df[TARGET_BIN_COL].values

print(f"Reference : {len(reference_df)} rows (train split)")
print(f"Analysis  : {len(analysis_df)} rows (test split)")
print(f"\nThe analysis set has NO target column — simulating production conditions.")

Reference : 7397 rows (train split)
Analysis  : 1874 rows (test split)

The analysis set has NO target column — simulating production conditions.


---
## 6. CBPE — Performance Estimation Without Ground Truth <a id='6-cbpe'></a>

**Confidence-Based Performance Estimation (CBPE)** works in two phases:

1. **Fit on reference:** Learn a calibrated mapping from the model's predicted probability to the realised performance metric (AUC, F1, etc.).
2. **Estimate on analysis:** Apply that mapping to the analysis set's probabilities. Since the relationship between probability and performance was learned from the reference, the estimate is valid *as long as the model's probability calibration hasn't changed* — i.e., as long as concept drift hasn't occurred.

The key insight: you don't need labels to get a useful performance estimate. You only need the model's output.

In [5]:
CHUNK_SIZE = 400  # number of rows per monitoring chunk

cbpe = nml.CBPE(
    y_pred_proba="y_pred_proba",
    y_pred="y_pred",
    y_true=TARGET_BIN_COL,
    timestamp_column_name="timestamp",
    metrics=["roc_auc", "f1"],
    chunk_size=CHUNK_SIZE,
    problem_type="classification_binary",
)
cbpe.fit(reference_df)
cbpe_results = cbpe.estimate(analysis_df)

cbpe_fig = cbpe_results.plot(kind="performance", metric="roc_auc")
cbpe_fig.show()

# Compare estimated vs actual
results_df = cbpe_results.filter(period="analysis").to_df()
estimated_auc = float(results_df[("roc_auc", "value")].mean())
print(f"Estimated AUC (CBPE) : {estimated_auc:.4f}")
print(f"Actual    AUC        : {actual_auc:.4f}")
print(f"CBPE error           : {abs(actual_auc - estimated_auc):.4f}")

Estimated AUC (CBPE) : 0.9647
Actual    AUC        : 0.9166
CBPE error           : 0.0481


**Interpreting the plot:**
- The solid line is the estimated AUC per chunk
- The shaded band is the confidence interval
- The horizontal dashed line is the reference AUC
- Points outside the band (flagged in red) signal estimated performance degradation

Since the test data is drawn from the same distribution as training, the estimated AUC should be close to the actual AUC — which validates that CBPE works on this dataset.

---
## 7. Univariate Drift Detection <a id='7-univariate'></a>

NannyML tests each feature independently:
- **Continuous features:** Kolmogorov-Smirnov test (distribution shape)
- **Categorical features:** Chi-squared test (category proportions)

The output is a p-value and an `alert` flag per chunk per feature. **Important caveat:** univariate tests can miss multivariate drift — e.g., if feature A and feature B individually look fine but their correlation structure has changed. That's why we run multivariate detection too.

In [6]:
univariate_calc = nml.UnivariateDriftCalculator(
    column_names=feature_cols,
    timestamp_column_name="timestamp",
    chunk_size=CHUNK_SIZE,
)
univariate_calc.fit(reference_df)
univariate_results = univariate_calc.calculate(analysis_df)

univariate_fig = univariate_results.plot(kind="drift")
univariate_fig.show()

# Summarise which features drifted
# Use positional access to avoid pandas tuple-key ambiguity
results_uv = univariate_results.filter(period="analysis").to_df()
cols_list = list(results_uv.columns)
alert_cols = list(dict.fromkeys(
    c[0] for i, c in enumerate(cols_list)
    if isinstance(c, tuple) and c[-1] == "alert" and c[0] in feature_cols
    and results_uv.iloc[:, i].any()
))
print(f"Drifted features ({len(alert_cols)} / {len(feature_cols)}): {alert_cols or 'none'}")

Drifted features (1 / 19): ['occupation']


---
## 8. Multivariate Drift Detection <a id='8-multivariate'></a>

NannyML's `DataReconstructionDriftCalculator` uses PCA to project the numeric features into a lower-dimensional space, then measures how well the reference distribution reconstructs the analysis data. A spike in reconstruction error = the joint distribution has shifted, even if individual features look stable.

In [7]:
multivariate_calc = nml.DataReconstructionDriftCalculator(
    column_names=numeric_cols,   # PCA works on numeric features only
    timestamp_column_name="timestamp",
    chunk_size=CHUNK_SIZE,
)
multivariate_calc.fit(reference_df)
multivariate_results = multivariate_calc.calculate(analysis_df)

multivariate_fig = multivariate_results.plot()
multivariate_fig.show()

---
## 9. Simulating Drift <a id='9-simulation'></a>

The test data above is drawn from the same distribution as training, so drift is minimal. Let's *simulate* what a macroeconomic shock looks like: compress `hours_per_week` (people shift to part-time work) and add noise.

In [8]:
analysis_drifted = analysis_df.copy()

if "hours_per_week" in analysis_drifted.columns:
    rng = np.random.RandomState(SEED)
    hpw = pd.to_numeric(analysis_drifted["hours_per_week"], errors="coerce")
    analysis_drifted["hours_per_week"] = (
        hpw * 0.65   # compress toward fewer hours
        + rng.normal(0, 4, len(analysis_drifted))    # add noise
    ).clip(1, 99)
    print("hours_per_week before perturbation:")
    print(f"  mean={pd.to_numeric(analysis_df['hours_per_week'], errors='coerce').mean():.1f}  std={pd.to_numeric(analysis_df['hours_per_week'], errors='coerce').std():.1f}")
    print("hours_per_week after perturbation:")
    print(f"  mean={analysis_drifted['hours_per_week'].mean():.1f}  std={analysis_drifted['hours_per_week'].std():.1f}")
else:
    print("hours_per_week not found in feature set — skipping perturbation")

hours_per_week before perturbation:
  mean=40.8  std=15.7
hours_per_week after perturbation:
  mean=26.7  std=10.9


In [12]:
# Re-run univariate detection on drifted data
univariate_calc_d = nml.UnivariateDriftCalculator(
    column_names=feature_cols,
    timestamp_column_name="timestamp",
    chunk_size=CHUNK_SIZE,
)
univariate_calc_d.fit(reference_df)
univariate_results_d = univariate_calc_d.calculate(analysis_drifted)

results_d = univariate_results_d.filter(period="analysis").to_df()
cols_list_d = list(results_d.columns)
drifted_after = list(dict.fromkeys(
    c[0] for i, c in enumerate(cols_list_d)
    if isinstance(c, tuple) and c[-1] == "alert" and c[0] in feature_cols
    and results_d.iloc[:, i].any()
))

univariate_results_d.filter(column_names=["hours_per_week"]).plot(kind="drift").show()

print(f"Before perturbation — drifted features: {alert_cols or 'none'}")
print(f"After  perturbation — drifted features: {drifted_after or 'none'}")

Before perturbation — drifted features: none
After  perturbation — drifted features: ['hours_per_week', 'occupation']


---
## 10. Logging Reports to MLflow <a id='10-mlflow'></a>

In production, monitoring runs on a schedule (nightly cron or a CI/CD trigger). Each run logs its reports to MLflow as artifacts, giving you a time-series of monitoring snapshots that live alongside your model versions. This is the connective tissue between MLflow and NannyML.

In [10]:
with mlflow.start_run(run_name="nannyml-monitoring-demo") as run:
    # Log scalar summary metrics
    mlflow.log_metric("estimated_val_auc", estimated_auc)
    mlflow.log_metric("actual_val_auc", actual_auc)
    mlflow.log_metric("n_drifted_features", len(alert_cols))
    mlflow.log_metric("cbpe_error", abs(actual_auc - estimated_auc))

    with tempfile.TemporaryDirectory() as tmpdir:
        # CBPE HTML report
        cbpe_path = Path(tmpdir) / "cbpe_report.html"
        cbpe_results.plot(kind="performance", metric="roc_auc").write_html(str(cbpe_path))
        mlflow.log_artifact(str(cbpe_path), artifact_path="nannyml_reports")

        # Drift HTML report
        drift_path = Path(tmpdir) / "drift_report.html"
        univariate_results.plot(kind="drift").write_html(str(drift_path))
        mlflow.log_artifact(str(drift_path), artifact_path="nannyml_reports")

        # Drift summary JSON
        summary_path = Path(tmpdir) / "drift_summary.json"
        summary = {"drifted_features": alert_cols, "n_drifted": len(alert_cols)}
        summary_path.write_text(json.dumps(summary, indent=2))
        mlflow.log_artifact(str(summary_path), artifact_path="nannyml_reports")

    monitoring_run_id = run.info.run_id
    print(f"Monitoring run logged. Run ID: {monitoring_run_id}")
    print(f"Open {MLFLOW_URI} → experiment '{MONITORING_EXPERIMENT}' to see the artifacts.")

Monitoring run logged. Run ID: 00e74f822dfe49c7a4f820f805d82e0a
Open http://127.0.0.1:5000 → experiment 'adult-income-monitoring' to see the artifacts.
🏃 View run nannyml-monitoring-demo at: http://127.0.0.1:5000/#/experiments/2/runs/00e74f822dfe49c7a4f820f805d82e0a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


---
## 11. Acceptance Checks <a id='11-checks'></a>

In [11]:
# ── Reference and analysis sets ───────────────────────────────────────────────
assert len(reference_df) > 0, "Reference set is empty"
assert len(analysis_df) > 0, "Analysis set is empty"
assert "y_pred_proba" in reference_df.columns, "Missing y_pred_proba in reference"
assert "y_pred_proba" in analysis_df.columns, "Missing y_pred_proba in analysis"
assert "timestamp" in reference_df.columns, "Missing timestamp in reference"

# ── CBPE quality ──────────────────────────────────────────────────────────────
cbpe_error = abs(actual_auc - estimated_auc)
assert cbpe_error < 0.10, f"CBPE error {cbpe_error:.4f} > 0.10 — check calibration"

# ── MLflow artifact logged ────────────────────────────────────────────────────
client = mlflow.tracking.MlflowClient()
artifacts = client.list_artifacts(monitoring_run_id, path="nannyml_reports")
artifact_names = [a.path for a in artifacts]
assert any("cbpe_report" in n for n in artifact_names), "CBPE report not found in MLflow"
assert any("drift_report" in n for n in artifact_names), "Drift report not found in MLflow"

print("✓ Reference and analysis sets built correctly")
print(f"✓ CBPE error: {cbpe_error:.4f} (within 0.10)")
print(f"✓ NannyML reports logged to MLflow run {monitoring_run_id}")
print("\nAll acceptance checks passed. 🎉")

✓ Reference and analysis sets built correctly
✓ CBPE error: 0.0481 (within 0.10)
✓ NannyML reports logged to MLflow run 00e74f822dfe49c7a4f820f805d82e0a

All acceptance checks passed. 🎉
